In [1]:
import requests
from minsearch import Index
from openai import OpenAI


docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses = response.json()

documents = []
url_prefix = "https://datatalks.club/faq/"
for course in courses:
    course_url = f"""{url_prefix}{course['path']}"""
    course_response = requests.get(course_url)
    course_data = course_response.json()
    documents.extend(course_data)

#print(documents[0])

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

question = "I just discoved this course. Can I join this now?"

def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2, "section": 0.5, "answer": 1}
    filter_dict = {"course": course}
    return index.search(
        question,
        boost_dict = boost_dict, 
        filter_dict = filter_dict,
        num_results=5
    )

search_results = search(question)

#print("Search results:",search_results)

def build_context(search_results):
    lines = []
    for result in search_results:
        lines.append("Q: " + result["section"])
        lines.append("A: " + result["answer"])
        lines.append("")  # Add a blank line between Q&A pairs

    return "\n".join(lines)


INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()



prompt = build_prompt(question, search_results)

##print(prompt)

openai_client = OpenAI()

#response = openai_client.responses.create(
#    model="gpt-5.4-mini",
#    input=prompt
#)

def llm(instructions, prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )
    return response.output_text

def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

answer = rag("I just discovered the course. Can I join now?")

print("Answer --- " + answer)

###Answer --- Yes — you can join now.

###You don’t need to register to start learning or submitting homework, as long as the submission form is still open. If you want a certificate, make sure to submit your project before submissions close.
###Usage --- ResponseUsage(input_tokens=425, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=50, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=475)


Answer --- Yes, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.
